# Demo

### Load Data and generate train/test splits

In [ ]:
import pandas as pd

from cyst_classifier.data_utils import generate_train_test_split

df = pd.read_csv("../data/kits_features.csv")
train_idx, test_idx = generate_train_test_split(df, test_size=0.2, random_state=42)
df.iloc[train_idx].to_csv("features_train.csv", index=False)
df.iloc[test_idx].to_csv("features_test.csv", index=False)

Found 'case' column in dataframe. Performed patient-stratified splitting.


## Train Models

In [2]:
! cyst-classifier train --data features_train.csv --model logistic --output-dir log --explain

Loading data from features_train.csv...
Using 10 features: ['mean_hu', 'std_hu', 'coefficient_of_variation', 'percentile_10', 'percentile_90', 'entropy', 'glcm_contrast', 'gradient_magnitude', 'sphericity', 'fraction_below_20hu']

✓ Detected pre-extracted feature CSV (fast mode)

Dataset summary:
  Total lesions: 975
  Tumors (label=2): 417
  Cysts (label=3): 558

Computing feature correlations...

Highly correlated feature pairs (|r| > 0.85):
  mean_hu <-> percentile_10: r = 0.864
  mean_hu <-> percentile_90: r = 0.921

Consider removing one feature from each highly correlated pair.

Training logistic model...

Model saved to log\model.pkl

Generating model explanations...

LOGISTIC REGRESSION MODEL EXPLANATION

Sample size: 975 lesions

Feature Analysis:
----------------------------------------------------------------------
Feature                  Coef   Odds Ratio               95% CI      P-adj   Sig
----------------------------------------------------------------------
mean_hu   

In [3]:
! cyst-classifier train --data features_train.csv --model tree --output-dir tree --explain --tree-depth 4

Loading data from features_train.csv...
Using 10 features: ['mean_hu', 'std_hu', 'coefficient_of_variation', 'percentile_10', 'percentile_90', 'entropy', 'glcm_contrast', 'gradient_magnitude', 'sphericity', 'fraction_below_20hu']

✓ Detected pre-extracted feature CSV (fast mode)

Dataset summary:
  Total lesions: 975
  Tumors (label=2): 417
  Cysts (label=3): 558

Computing feature correlations...

Highly correlated feature pairs (|r| > 0.85):
  mean_hu <-> percentile_10: r = 0.864
  mean_hu <-> percentile_90: r = 0.921

Consider removing one feature from each highly correlated pair.

Training tree model...

Model saved to tree\model.pkl

Generating model explanations...

DECISION TREE MODEL EXPLANATION

Tree depth: 4
Number of leaves: 14
Features used: 7 / 10

Feature Importance:
----------------------------------------------------------------------
Feature                     Importance                  Bar
----------------------------------------------------------------------
glcm_c

## Evaluate Models

In [4]:
# Evaluate on cached features (fast)
! cyst-classifier eval --data features_test.csv --model log/model.pkl --output-dir log --uncertainty-threshold 0.60 --explain

Loading model from log/model.pkl...
Loading test data from features_test.csv...

✓ Detected pre-extracted feature CSV (fast mode)
Running inference on test set...

Evaluation summary:
  Total lesions: 214
  Tumors (label=2): 104
  Cysts (label=3): 110

Using uncertainty threshold: 0.6

CLASSIFICATION METRICS
(excluding unsure predictions)
Accuracy:    0.9171
F1 Score:    0.9158
Sensitivity: 0.9158
Specificity: 0.9184
AUROC:       0.9443

Coverage:    0.9019 (193/214 certain)
Unsure:      21 predictions

Confusion Matrix (certain predictions only):
                Predicted
              Tumor  Cyst
True Tumor       90      8
     Cyst         8     87

Full Confusion Matrix (including unsure):
                     Predicted
              Tumor  Cyst  Unsure
True Tumor       90      8       6
     Cyst         8     87      15

ROC curve saved to log\roc_curve.png
Confusion matrix saved to log\confusion_matrix.png

Results saved to log

Generating model explanations...

LOGISTIC REGRESS

In [5]:
! cyst-classifier eval --data features_test.csv --model tree/model.pkl --output-dir tree --uncertainty-threshold 0.60 --explain 

Loading model from tree/model.pkl...
Loading test data from features_test.csv...

✓ Detected pre-extracted feature CSV (fast mode)
Running inference on test set...

Evaluation summary:
  Total lesions: 214
  Tumors (label=2): 104
  Cysts (label=3): 110

Using uncertainty threshold: 0.6

CLASSIFICATION METRICS
(excluding unsure predictions)
Accuracy:    0.8775
F1 Score:    0.8826
Sensitivity: 0.8952
Specificity: 0.8586
AUROC:       0.9253

Coverage:    0.9533 (204/214 certain)
Unsure:      10 predictions

Confusion Matrix (certain predictions only):
                Predicted
              Tumor  Cyst
True Tumor       85     14
     Cyst        11     94

Full Confusion Matrix (including unsure):
                     Predicted
              Tumor  Cyst  Unsure
True Tumor       85     14       5
     Cyst        11     94       5

ROC curve saved to tree\roc_curve.png
Confusion matrix saved to tree\confusion_matrix.png

Results saved to tree

Generating model explanations...
  Identified 